# SQL joins: inner, left, anti and semi

A join combines rows from two tables that match on a key. INNER keeps matches only, LEFT keeps every left row and fills NULLs where nothing matches, LEFT ANTI keeps left rows with no match and LEFT SEMI keeps left rows that have a match. Duplicate keys multiply rows.

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end.

From the Learning Hub lesson: [SQL joins: inner, left, anti and semi](https://www.datapreplab.com/#topic/sql-joins) · Independent study resource, not affiliated with or endorsed by Databricks.

**Goal:** see each join type on the same data and spot the orphan row.

**Part A: data.**

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES (1, 'Asha'), (2, 'Ben'), (3, 'Chen') AS t(id, name);

CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES (10, 1), (11, 1), (12, 3), (13, 4) AS t(order_id, customer_id);

**Part B: compare join types.** Predict each row count first:

In [ ]:
%sql
SELECT count(*) FROM customers c JOIN orders o ON c.id = o.customer_id;          -- 3
SELECT count(*) FROM customers c LEFT JOIN orders o ON c.id = o.customer_id;     -- 4
SELECT count(*) FROM customers c FULL JOIN orders o ON c.id = o.customer_id;     -- 5

**Part C: orders per customer, including zero.**

In [ ]:
%sql
SELECT c.id, c.name, count(o.order_id) AS orders
FROM customers c LEFT JOIN orders o ON c.id = o.customer_id
GROUP BY c.id, c.name ORDER BY c.id;

**Part D: anti joins both ways.**

In [ ]:
%sql
SELECT * FROM customers c LEFT ANTI JOIN orders o ON c.id = o.customer_id;   -- Ben
SELECT * FROM orders o LEFT ANTI JOIN customers c ON o.customer_id = c.id;   -- order 13

**Part E: watch rows multiply.** Add a duplicate customer and rerun the INNER count:

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES (1, 'Asha'), (1, 'Asha'), (2, 'Ben'), (3, 'Chen') AS t(id, name);
SELECT count(*) FROM customers c JOIN orders o ON c.id = o.customer_id;      -- 5

Expected evidence:

```text
Part B -> 3, 4, 5
Part C -> 1 Asha 2 ; 2 Ben 0 ; 3 Chen 1
Part D -> Ben ; order 13
Part E -> 5 (Asha's two orders now appear twice)
```